# Imports

In [35]:
%pip install catboost lightgbm optuna holidays

In [36]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
%matplotlib inline

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, TimeSeriesSplit, cross_val_predict, cross_val_score, cross_validate
from sklearn.preprocessing import TargetEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.base import clone
from sklearn.compose import ColumnTransformer

from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error

import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor, Pool

import optuna
from typing import Any
import warnings

import holidays

In [37]:
import plotly.io as pio
pio.renderers.default = "vscode"

# Global Vars

In [38]:
TARGET = "num_sold"
N_SPLITS = 5
SEED = 42
eps = 1e-5

def check_environment():
    try:
        import google.colab
        env = "colab"
    except ImportError:
        env = "local"
    return env

ENVIRONMENT = check_environment()

non_relevant_columns = ['id', 'source']

# ordinal_maps = {
#     "gender" : {"male":0, "female":1, "other":2},
#     "internet_access" : {"no":0, "yes":1},
#     "sleep_quality" : {"poor":0, "average":1, "good":2},
#     "facility_rating" : {"low":0, "medium":1, "high":2},
#     "exam_difficulty" : {"easy":0, "moderate":1, "hard":2},
#     "course" : {"ba":0, "b.sc":1, "diploma":2, "b.tech":3, "b.com":4, "bca":5, "bba":6},
#     "study_method" : {"self-study":0, "online videos":1, "group study":2, "mixed":3, "coaching":4},
# }

original_categorical_cols = ['country', 'store', 'product']


# Load Data

In [39]:
def load_data(option: str="local"):
  '''
  returns train_df, test_df, df

  '''
  if option == "local":
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

  elif option == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    train_df = pd.read_csv('/content/drive/MyDrive/Kaggle Practice/Sticker Sales Forecast/train.csv')
    test_df = pd.read_csv('/content/drive/MyDrive/Kaggle Practice/Sticker Sales Forecast/test.csv')

  train_df['source'] = 'train'
  test_df['source'] = 'test'
  train_df['date'] =  pd.to_datetime(train_df['date'])
  test_df['date'] =  pd.to_datetime(test_df['date'])
  df = pd.concat([train_df, test_df], ignore_index=True)

  return train_df, test_df, df

# Data Splitter

In [40]:
def data_spliter(df: pd.DataFrame, additional_drop_columns: list = []) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    '''
    return: X_train, y_train, submission_df

    '''
    train_dataset = df[df['source'] == 'train'].drop(columns=non_relevant_columns + additional_drop_columns)
    submission_dataset = df[df['source'] == 'test'].drop(columns=['source', TARGET] + additional_drop_columns)

    X_train = train_dataset.drop(columns=[TARGET])
    y_train = train_dataset[TARGET]

    return X_train, y_train, submission_dataset

# EDA

In [41]:
df_train, df_test, df = load_data(option=ENVIRONMENT)

KeyboardInterrupt: 

## Info / describe / etc.

In [0]:
df.head()

In [0]:
df.info()

In [0]:
df.describe()

In [0]:
df.isna().sum()

## Test vs Train date

In [0]:
sns.histplot(df, x='date', hue='source')

## Num Sold overtime

In [0]:
df['date'].dt.year

In [0]:
plt.figure(figsize=(10,6))
hist_data = df_train.copy()
hist_data['year'] = hist_data['date'].dt.year
sns.histplot(
    data=hist_data,
    x='num_sold',
    hue='year',
    bins=30,
    log_scale=True,
    palette='viridis',
    multiple='fill',
)

In [0]:
sns.lineplot(data=df, x='date', y='num_sold')

In [0]:
plt.figure(figsize=(10,6))
grouped = df_train.groupby([pd.Grouper(key='date', freq='D'), 'product'])['num_sold'].sum().reset_index()
sns.lineplot(data=grouped, x='date', y='num_sold', hue='product')

In [0]:
grouped = df_train.groupby([pd.Grouper(key='date', freq='MS'), 'country'])['num_sold'].sum().reset_index()
sns.lineplot(data=grouped, x='date', y='num_sold', hue='country')

In [0]:
df['product'].unique()

## Country

In [0]:
df['country'].unique()

In [0]:
holidays.KE( years=range(2020, 2027))

# Feature engineering

In [ ]:
def feature_engineering(fdf: pd.DataFrame) -> pd.DataFrame:
    '''
    return: df with new features

    '''
    df = fdf.copy()
    df = df[~((df['source'] == 'train') & (df['num_sold'].isna()))].reset_index(drop=True)
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['quarter'] = df['date'].dt.quarter
    df['day_of_week'] = df['date'].dt.dayofweek
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    
    # Add holiday feature for Kenya

    return df

# Custom yearly splitter

In [ ]:
class YearSplitter:
    def __init__(self, year_col:str = 'year', horizon: int = 1, min_train_years:int = 1):
        if horizon < 1:
            raise ValueError(f"horizon should be > 0, got {horizon}")
        if min_train_years < 1:
            raise ValueError(f"min_train_years should be > 0, got {min_train_years}")
        self.year_col = year_col
        self.horizon = horizon
        self.min_train_years = min_train_years
    
    def split(self, X):
        years = np.sort(X[self.year_col].unique())
        split_years = years[self.min_train_years - 1 : len(years) - self.horizon]
        if len(split_years) == 0:
            raise ValueError(f"Not enough years for min_train_year = {self.min_train_years} and horizon = {self.horizon}")
        for split_year in split_years:
            tr_idx = np.where(X[self.year_col] <= split_year)[0]
            val_idx = np.where((X[self.year_col] > split_year) & (X[self.year_col] <= split_year + self.horizon))[0]
            yield tr_idx, val_idx

# Catboost

In [42]:
_, _, df = load_data(ENVIRONMENT)
df = feature_engineering(df)
df[['country', 'store', 'product']] = df[['country', 'store', 'product']].astype('str')
X, y, submission_df = data_spliter(df, additional_drop_columns=['date', 'year'])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [47]:
ycv = YearSplitter(horizon=1, min_train_years=1)
train_metric_scores, valid_metric_scores, best_iterations = [], [], []
split_df = df[df['source'] == 'train']

for fold, (train_idx, val_idx) in enumerate(ycv.split(split_df)):
    print(f"\n ======= TRAINING FOLD: {fold + 1} =======")
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    train_pool = Pool(X.iloc[train_idx], y_train, cat_features=original_categorical_cols)
    val_pool = Pool(X.iloc[val_idx], y_val, cat_features=original_categorical_cols)
    # X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    # y_train_log, y_val_log = y_log.iloc[train_idx], y_log.iloc[val_idx]
    # y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=5000,
        learning_rate=0.05,
        depth=5,
        l2_leaf_reg=3,
        random_strength=10,
        random_seed=SEED,
        bootstrap_type='Bernoulli',
        subsample=0.75,
        one_hot_max_size=2
        
        # device="gpu"
    )
    model.fit(
        train_pool,
        eval_set=val_pool,
        verbose=500,
        early_stopping_rounds=100
    )

    train_prediction= model.predict(train_pool)
    validation_prediction= model.predict(val_pool)
    # train_prediction = np.expm1(train_prediction_log)
    # validation_prediction = np.expm1(validation_prediction_log)
    # train_prediction = np.clip(train_prediction, 0, None)
    # validation_prediction = np.clip(validation_prediction, 0, None)

    # oof[val_idx] = validation_prediction
    best_iterations.append(model.best_iteration_)

    train_metric = mean_absolute_percentage_error(y_train, train_prediction)
    validation_metric = mean_absolute_percentage_error(y_val, validation_prediction)

    train_metric_scores.append(train_metric)
    valid_metric_scores.append(validation_metric)

    # results_lgb += model.predict(X_sub_raw) / N_SPLITS

    print(f"======= FOLD: {fold + 1} =======")
    print(f"TRAIN MAPE {train_metric} | VALID MAPE {validation_metric}\n | DELTA: {validation_metric - train_metric} | BEST ITER: {model.best_iteration_}")

tr, va = np.array(train_metric_scores), np.array(valid_metric_scores)
print("\n====== CROSS VALIDATION ENDED ======")
print(f"TRAIN AVG MAPE {tr.mean():.5f} +/- {tr.std():.5f} | "
      f"VAL AVG MAPE {va.mean():.5f} +/- {va.std():.5f}")
print(f"MEAN BEST ITER: {np.mean(best_iterations):.0f}")
# print(f"\nOOF MAPE {root_mean_squared_error(y, oof_lgb):.5f}")



 ======= TRAINING FOLD: 1 =======
0:	learn: 624.1111591	test: 771.5939624	best: 771.5939624 (0)	total: 16.1ms	remaining: 1m 20s
500:	learn: 70.8450590	test: 215.7859790	best: 215.7560092 (499)	total: 7.97s	remaining: 1m 11s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 214.305528
bestIteration = 873

Shrink model to first 874 iterations.
======= FOLD: 1 =======
TRAIN MAPE 0.18867340276502192 | VALID MAPE 0.2640249145847814
 | DELTA: 0.07535151181975946 | BEST ITER: 873

 ======= TRAINING FOLD: 2 =======
0:	learn: 674.0214524	test: 693.2321116	best: 693.2321116 (0)	total: 34ms	remaining: 2m 49s
500:	learn: 127.7926662	test: 104.4446990	best: 104.4418501 (499)	total: 12.7s	remaining: 1m 54s
1000:	learn: 122.0563370	test: 101.3254260	best: 101.3174910 (997)	total: 27.2s	remaining: 1m 48s
1500:	learn: 118.9638300	test: 100.0087860	best: 99.9908769 (1497)	total: 43.2s	remaining: 1m 40s
2000:	learn: 116.8839243	test: 99.4861145	best: 99.4777548 (1993)	total: 1m 1s	remai

KeyboardInterrupt: 

# LGBM

In [32]:
_, _, df = load_data(ENVIRONMENT)
df = feature_engineering(df)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [33]:
df[['country', 'store', 'product']] = df[['country', 'store', 'product']].astype('category')

In [34]:
X, y, submission_df = data_spliter(df, additional_drop_columns=['date', 'year'])

In [22]:
y_log = np.log1p(y)

In [25]:
ycv = YearSplitter(horizon=1, min_train_years=1)
train_metric_scores, valid_metric_scores, best_iterations = [], [], []
split_df = df[df['source'] == 'train']
for fold, (train_idx, val_idx) in enumerate(ycv.split(split_df)):
    print(f"\n ======= TRAINING FOLD: {fold + 1} =======")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train_log, y_val_log = y_log.iloc[train_idx], y_log.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = lgb.LGBMRegressor(
        objective='regression',
        metric='rmse',
        n_estimators=5000,
        learning_rate=0.01,
        num_leaves=24,
        subsample=1,
        subsample_freq=1,
        colsample_bytree=1,
        random_state=SEED,
        verbose=-1,
        # device="gpu"
    )
    model.fit(
        X_train,
        y_train_log,
        categorical_feature=original_categorical_cols,
        eval_metric="rmse",
        # eval_X=X_val,
        # eval_y=y_val_log,
        eval_set=[(X_val, y_val_log)],
        callbacks=[lgb.early_stopping(stopping_rounds=100),
                   lgb.log_evaluation(period=500)]
    )

    train_prediction_log = model.predict(X_train)
    validation_prediction_log = model.predict(X_val)
    train_prediction = np.expm1(train_prediction_log)
    validation_prediction = np.expm1(validation_prediction_log)
    train_prediction = np.clip(train_prediction, 0, None)
    validation_prediction = np.clip(validation_prediction, 0, None)

    # oof[val_idx] = validation_prediction
    best_iterations.append(model.best_iteration_)

    train_metric = mean_absolute_percentage_error(y_train, train_prediction)
    validation_metric = mean_absolute_percentage_error(y_val, validation_prediction)

    train_metric_scores.append(train_metric)
    valid_metric_scores.append(validation_metric)

    # results_lgb += model.predict(X_sub_raw) / N_SPLITS

    print(f"======= FOLD: {fold + 1} =======")
    print(f"TRAIN MAPE {train_metric} | VALID MAPE {validation_metric}\n | DELTA: {validation_metric - train_metric} | BEST ITER: {model.best_iteration_}")

tr, va = np.array(train_metric_scores), np.array(valid_metric_scores)
print("\n====== CROSS VALIDATION ENDED ======")
print(f"TRAIN AVG MAPE {tr.mean():.5f} +/- {tr.std():.5f} | "
      f"VAL AVG MAPE {va.mean():.5f} +/- {va.std():.5f}")
print(f"MEAN BEST ITER: {np.mean(best_iterations):.0f}")
# print(f"\nOOF MAPE {root_mean_squared_error(y, oof_lgb):.5f}")



 ======= TRAINING FOLD: 1 =======
Training until validation scores don't improve for 100 rounds
[500]	valid_0's rmse: 0.175124
[1000]	valid_0's rmse: 0.164726
[1500]	valid_0's rmse: 0.163906
[2000]	valid_0's rmse: 0.163794
Early stopping, best iteration is:
[1996]	valid_0's rmse: 0.163786
======= FOLD: 1 =======
TRAIN MAPE 0.04319901267478823 | VALID MAPE 0.13124684114671362
 | DELTA: 0.08804782847192538 | BEST ITER: 1996

 ======= TRAINING FOLD: 2 =======
Training until validation scores don't improve for 100 rounds
[500]	valid_0's rmse: 0.119065
[1000]	valid_0's rmse: 0.106969
[1500]	valid_0's rmse: 0.105517
[2000]	valid_0's rmse: 0.10491
Early stopping, best iteration is:
[2008]	valid_0's rmse: 0.104907
======= FOLD: 2 =======
TRAIN MAPE 0.07472948525530013 | VALID MAPE 0.0840930556422403
 | DELTA: 0.009363570386940176 | BEST ITER: 2008

 ======= TRAINING FOLD: 3 =======
Training until validation scores don't improve for 100 rounds
[500]	valid_0's rmse: 0.119634
[1000]	valid_0's rm

In [27]:
model_lgb = lgb.LGBMRegressor(
        objective='regression',
        metric='rmse',
        n_estimators=1448,
        learning_rate=0.01,
        num_leaves=24,
        subsample=1,
        subsample_freq=1,
        colsample_bytree=1,
        random_state=SEED,
        verbose=-1,
        # device="gpu"
    )

model_lgb.fit(X, y_log, categorical_feature=original_categorical_cols)
lgb_pred_log = model_lgb.predict(submission_df.drop(columns='id'))
lgb_pred = np.expm1(lgb_pred_log)
lgb_pred = np.clip(lgb_pred, 0, None)

In [30]:
submission_df_lgb = submission_df.copy()
submission_df_lgb[TARGET] = lgb_pred
if ENVIRONMENT == "colab":
    submission_df_lgb[['id', TARGET]].to_csv('/content/drive/MyDrive/Kaggle Practice/Sticker Sales Forecast/lgb_submission_v3.csv', index=False)
else:
    submission_df_lgb[['id', TARGET]].to_csv('lgb_submission_v3.csv', index=False)

'colab'